In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/late_fusion_results"
os.makedirs(SAVE_DIR, exist_ok=True)

print("✅ Save directory ready:", SAVE_DIR)

✅ Save directory ready: /content/drive/MyDrive/late_fusion_results


In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    precision_score, recall_score, f1_score
)

In [ ]:
train_df = pd.read_csv("/content/drive/MyDrive/weighted_fusion_results/fusion_train.csv")
val_df   = pd.read_csv("/content/drive/MyDrive/weighted_fusion_results/fusion_val.csv")
test_df  = pd.read_csv("/content/drive/MyDrive/weighted_fusion_results/fusion_test.csv")

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Train: (20970, 11)
Val  : (4494, 11)
Test : (4494, 11)


In [ ]:
print(train_df.columns)

Index(['DR_prob', 'Gl_prob', 'AMD_prob', 'DED_prob', 'is_fundus', 'is_oct',
       'is_slitlamp', 'DR', 'Glaucoma', 'AMD', 'DED'],
      dtype='object')


In [ ]:
from sklearn.preprocessing import StandardScaler

FEATURES = [
    'DR_prob', 'Gl_prob', 'AMD_prob', 'DED_prob',
    'is_fundus', 'is_oct', 'is_slitlamp'
]

LABELS = ['DR', 'Glaucoma', 'AMD', 'DED']

X_train = train_df[FEATURES].values
y_train = train_df[LABELS].values

X_val = val_df[FEATURES].values
y_val = val_df[LABELS].values

X_test = test_df[FEATURES].values
y_test = test_df[LABELS].values

# 🔥 NORMALIZATION (NEW)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print("Feature shape:", X_train.shape)
print("Label shape:", y_train.shape)

Feature shape: (20970, 7)
Label shape: (20970, 4)


In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float()),
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val).float(), torch.tensor(y_val).float()),
    batch_size=256,
    shuffle=False
)

In [ ]:
class LateFusionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Linear(16, 4)
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
model = LateFusionMLP().to(DEVICE)

# 🔥 CLASS WEIGHTING (BOOST DR)
pos_weight = torch.tensor([2.0, 1.0, 1.0, 1.0]).to(DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
best_val_loss = float("inf")

for epoch in range(30):
    model.train()
    train_loss = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()

    print(f"Epoch {epoch+1}: Train={train_loss:.4f}, Val={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save({
            "model": model.state_dict()
        }, f"{SAVE_DIR}/late_fusion_best.pth")

        print("✅ Best model saved")

Epoch 1: Train=50.4157, Val=8.6921
✅ Best model saved
Epoch 2: Train=32.2427, Val=5.6110
✅ Best model saved
Epoch 3: Train=20.9830, Val=3.7465
✅ Best model saved
Epoch 4: Train=14.8296, Val=2.7387
✅ Best model saved
Epoch 5: Train=11.6873, Val=2.2432
✅ Best model saved
Epoch 6: Train=9.9380, Val=1.9539
✅ Best model saved
Epoch 7: Train=8.9664, Val=1.7751
✅ Best model saved
Epoch 8: Train=8.3796, Val=1.6738
✅ Best model saved
Epoch 9: Train=7.9248, Val=1.5732
✅ Best model saved
Epoch 10: Train=7.6064, Val=1.5184
✅ Best model saved
Epoch 11: Train=7.3504, Val=1.4699
✅ Best model saved
Epoch 12: Train=7.1808, Val=1.4440
✅ Best model saved
Epoch 13: Train=7.0336, Val=1.4006
✅ Best model saved
Epoch 14: Train=6.9621, Val=1.3836
✅ Best model saved
Epoch 15: Train=6.8120, Val=1.3610
✅ Best model saved
Epoch 16: Train=6.7895, Val=1.3551
✅ Best model saved
Epoch 17: Train=6.6330, Val=1.3271
✅ Best model saved
Epoch 18: Train=6.6089, Val=1.3162
✅ Best model saved
Epoch 19: Train=6.6163, Val=1.30

In [ ]:
ckpt = torch.load(f"{SAVE_DIR}/late_fusion_best.pth", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

LateFusionMLP(
  (net): Sequential(
    (0): Linear(in_features=7, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=32, out_features=16, bias=True)
    (5): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Linear(in_features=16, out_features=4, bias=True)
  )
)

In [ ]:
def get_predictions(model, X):
    model.eval()
    with torch.no_grad():
        x = torch.tensor(X).float().to(DEVICE)
        logits = model(x)
        probs = torch.sigmoid(logits).cpu().numpy()
    return probs

In [ ]:
thresholds = np.arange(0.1, 0.9, 0.05)

best_thresh = [0.5, 0.5, 0.5, 0.5]

val_probs = get_predictions(model, X_val)

for i in range(4):
    best_score = 0

    for t in thresholds:
        preds = (val_probs[:, i] > t).astype(int)

        prec = precision_score(y_val[:, i], preds, zero_division=0)
        rec  = recall_score(y_val[:, i], preds, zero_division=0)

        # 🔥 SPECIAL HANDLING FOR DR
        if i == 0:
            score = 0.7 * rec + 0.3 * prec
        else:
            score = f1_score(y_val[:, i], preds)

        if score > best_score:
            best_score = score
            best_thresh[i] = t

print("✅ Best thresholds:", best_thresh)

✅ Best thresholds: [np.float64(0.30000000000000004), np.float64(0.45000000000000007), np.float64(0.1), np.float64(0.3500000000000001)]


In [ ]:
test_probs = get_predictions(model, X_test)

results = []
disease_names = ["DR", "Glaucoma", "AMD", "DED"]

for i in range(4):
    preds = (test_probs[:, i] > best_thresh[i]).astype(int)

    auc = roc_auc_score(y_test[:, i], test_probs[:, i])
    acc = accuracy_score(y_test[:, i], preds)
    prec = precision_score(y_test[:, i], preds)
    rec = recall_score(y_test[:, i], preds)
    f1 = f1_score(y_test[:, i], preds)

    results.append([disease_names[i], auc, acc, prec, rec, f1])

df_results = pd.DataFrame(results,
    columns=["Disease", "AUC", "Accuracy", "Precision", "Recall", "F1"]
)

df_results.to_csv(f"{SAVE_DIR}/disease_metrics.csv", index=False)

df_results

,Disease,AUC,Accuracy,Precision,Recall,F1
0,DR,0.986803,0.953049,0.571429,0.974910,0.720530
1,Glaucoma,0.976675,0.937027,0.836408,0.819277,0.827754
2,AMD,0.999373,0.995327,0.990955,0.987976,0.989463
3,DED,0.999996,0.999555,0.965517,1.000000,0.982456


In [ ]:
layers = [4, 8, 16, 32, 64]
layer_results = []

for l in layers:

    class TempModel(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(7, l),
                nn.ReLU(),
                nn.Linear(l, 4)
            )

        def forward(self, x):
            return self.net(x)

    temp_model = TempModel().to(DEVICE)
    optimizer = torch.optim.Adam(temp_model.parameters(), lr=1e-3)

    for epoch in range(5):
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()
            loss = criterion(temp_model(x), y)
            loss.backward()
            optimizer.step()

    probs = get_predictions(temp_model, X_val)
    auc = roc_auc_score(y_val, probs, average="macro")

    layer_results.append([l, auc])

layer_df = pd.DataFrame(layer_results, columns=["Hidden Units", "Val AUC"])
layer_df.to_csv(f"{SAVE_DIR}/layer_comparison.csv", index=False)

layer_df

,Hidden Units,Val AUC
0,4,0.936363
1,8,0.980568
2,16,0.987531
3,32,0.988623
4,64,0.989417


In [ ]:
np.save(f"{SAVE_DIR}/test_probs.npy", test_probs)
np.save(f"{SAVE_DIR}/test_labels.npy", y_test)

print("✅ Predictions saved")

✅ Predictions saved


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# 1. Logistic Regression
# =========================
logreg_preds = []

for i in range(4):
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train[:, i])
    probs = clf.predict_proba(X_test)[:, 1]
    logreg_preds.append(probs)

logreg_preds = np.stack(logreg_preds, axis=1)

# =========================
# 2. 1-Layer MLP
# =========================
class MLP1(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        return self.net(x)

model1 = MLP1().to(DEVICE)
opt1 = torch.optim.Adam(model1.parameters(), lr=1e-3)
crit = nn.BCEWithLogitsLoss()

for epoch in range(10):
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt1.zero_grad()
        loss = crit(model1(x), y)
        loss.backward()
        opt1.step()

with torch.no_grad():
    probs1 = torch.sigmoid(model1(torch.tensor(X_test).float().to(DEVICE))).cpu().numpy()

# =========================
# 3. 2-Layer MLP
# =========================
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)
        )

    def forward(self, x):
        return self.net(x)

model2 = MLP2().to(DEVICE)
opt2 = torch.optim.Adam(model2.parameters(), lr=1e-3)

for epoch in range(10):
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt2.zero_grad()
        loss = crit(model2(x), y)
        loss.backward()
        opt2.step()

with torch.no_grad():
    probs2 = torch.sigmoid(model2(torch.tensor(X_test).float().to(DEVICE))).cpu().numpy()

# =========================
# 4. Compute AUCs
# =========================
diseases = ["DR", "Glaucoma", "AMD", "DED"]

rows = []

for i in range(4):
    auc_lr = roc_auc_score(y_test[:, i], logreg_preds[:, i])
    auc_1  = roc_auc_score(y_test[:, i], probs1[:, i])
    auc_2  = roc_auc_score(y_test[:, i], probs2[:, i])

    rows.append([diseases[i], auc_lr, auc_1, auc_2])

# =========================
# 5. Macro AUC
# =========================
macro_lr = roc_auc_score(y_test, logreg_preds, average="macro")
macro_1  = roc_auc_score(y_test, probs1, average="macro")
macro_2  = roc_auc_score(y_test, probs2, average="macro")

rows.append(["Macro AUC", macro_lr, macro_1, macro_2])

# =========================
# 6. Final Table
# =========================
df_compare = pd.DataFrame(
    rows,
    columns=["Disease", "Logistic Regression", "1-Layer MLP", "2-Layer MLP"]
)

# Save
df_compare.to_csv(f"{SAVE_DIR}/model_comparison.csv", index=False)

df_compare

,Disease,Logistic Regression,1-Layer MLP,2-Layer MLP
0,DR,0.987712,0.977927,0.983464
1,Glaucoma,0.973967,0.974787,0.975652
2,AMD,0.999373,0.999373,0.999373
3,DED,0.999996,0.999996,0.999996
4,Macro AUC,0.990262,0.988021,0.989621
